### Generating text with a pretrained llm

In [1]:
import sys
sys.path.append('..')
from downloading_the_base_model.download_model import downlaod_model
downlaod_model(
    repo_id="Qwen/Qwen3-0.6B-Base",
    local_dir="qwen",
)

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 508.37it/s]


'C:\\Users\\Admin\\Documents\\open-posttraining-system\\generating_text_with_pre-trained_llm\\qwen'

In [2]:
from pathlib import Path
from base_model.qwen import Qwen3Tokenizer

tokenizer_path = Path("qwen") / "tokenizer.json"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [3]:
tokenizer

In [4]:
prompt = "Explain neural networks in simple terms."
token_list = tokenizer.encode(prompt=prompt)
token_list

[840, 20772, 29728, 14155, 304, 4285, 3793, 13]

In [5]:
for i in token_list:
    print(f"{i} ---> {tokenizer.decode([i])}")

840 ---> Ex
20772 ---> plain
29728 --->  neural
14155 --->  networks
304 --->  in
4285 --->  simple
3793 --->  terms
13 ---> .


In [6]:
text = tokenizer.decode(token_list)
print(text)

Explain neural networks in simple terms.


### loading the pretraind model

In [3]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU")

GPU available: NVIDIA GeForce RTX 3050 Laptop GPU


In [8]:
# Diagnostic info
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("To use GPU, you need to:")
    print("1. Have an NVIDIA GPU installed")
    print("2. Install NVIDIA CUDA drivers from https://developer.nvidia.com/cuda-downloads")
    print("3. Reinstall PyTorch with CUDA support: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
from pathlib import Path
from safetensors.torch import load_file
from base_model.qwen import Qwen3Model, load_hf_weights_into_qwen, QWEN_CONFIG_06_B

weights_path = Path("qwen") / "model.safetensors"

# Load ONCE
weights = load_file(weights_path)

model = Qwen3Model(QWEN_CONFIG_06_B)

load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN_CONFIG_06_B["n_layers"],
        "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"]
    },
    params=weights
)

model.to("cuda" if torch.cuda.is_available() else "cpu")
model.to(torch.bfloat16)
model.eval()

Qwen3Model(
  (emb): Embedding(151936, 1024)
  (t_block): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=True)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=True)
        (w_values): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=True)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

### ok, now understanding the sequential llm text generation process.

In [25]:
prompt = "Explain large language models."
list_of_encoded_tokens= tokenizer.encode(prompt)
print(f"list_of_encoded_tokens: {list_of_encoded_tokens}")
list_of_encoded_tokens_tensor = torch.tensor(list_of_encoded_tokens)
print(f"list_of_encoded_tokens_tensor shape: {list_of_encoded_tokens_tensor.shape}")
print(f"list_of_encoded_tokens_tensor dtype: {list_of_encoded_tokens_tensor.dtype}")

encoded_tokens_fmt=list_of_encoded_tokens_tensor.unsqueeze(0).to(device)
print(f"encoded_tokens_fmt shape: {encoded_tokens_fmt.shape}")
print(f"encoded_tokens_fmt dtype: {encoded_tokens_fmt.dtype}")
print(f"model dtype: {next(model.parameters()).dtype}")
print(f"model device: {next(model.parameters()).device}")

##-----
with torch.inference_mode():
    output_tensor = model(encoded_tokens_fmt)
    print(output_tensor.shape)
    
output_tensor_fmt = output_tensor.squeeze(0)
print(f"shape of output_tensor_fmt: {output_tensor_fmt.shape}")

list_of_encoded_tokens: [840, 20772, 3460, 4128, 4119, 13]
list_of_encoded_tokens_tensor shape: torch.Size([6])
list_of_encoded_tokens_tensor dtype: torch.int64
encoded_tokens_fmt shape: torch.Size([1, 6])
encoded_tokens_fmt dtype: torch.int64
model dtype: torch.bfloat16
model device: cuda:0
torch.Size([1, 6, 151936])
shape of output_tensor_fmt: torch.Size([6, 151936])


In [26]:
last_token = output_tensor_fmt[-1]
print(last_token)
print(last_token.shape)

tensor([-4.7812, -7.5312,  1.1562,  ...,  3.6875,  3.6875,  3.6875],
       device='cuda:0', dtype=torch.bfloat16)
torch.Size([151936])


In [30]:
next_token = torch.argmax(last_token, dim=-1, keepdim=True)

print(next_token)
print(next_token.shape)

tensor([40717], device='cuda:0')
torch.Size([1])


In [31]:
decoded_next_token = tokenizer.decode([40717])
print(decoded_next_token)


kor


In [40]:
# DETAILED GENERATION QUALITY CHECK
print("="*80)
print("GENERATION QUALITY & BASE MODEL BEHAVIOR CHECK")
print("="*80)

# 1. Check tokenizer filename detection
print("\n1. TOKENIZER EOS TOKEN ISSUE")
print("-" * 80)
tokenizer_path_str = str(tokenizer_path)
print(f"Tokenizer path: {tokenizer_path_str}")
fname = tokenizer_path_str.split('\\')[-1].lower()
print(f"Filename lowercase: {fname}")
print(f"Contains 'base': {'base' in fname}")
print(f"Contains 'reasoning': {'reasoning' in fname}")
print(f"EOS should be '<|endoftext|>' for base, but got: '{tokenizer.eos_token}'")
print(f"Issue: The filename detection logic may be broken")

# 2. Compare with actual Qwen3 model behavior
print("\n2. BASE MODEL GENERATION EXPECTATIONS")
print("-" * 80)
print(f"Model: Qwen 3 - 0.6B Base (NOT instruction-tuned)")
print(f"Characteristics:")
print(f"  - Trained on raw text/next-token prediction only")
print(f"  - Does NOT follow instructions")
print(f"  - Generates statistically likely continuations")
print(f"  - May generate random subword tokens or untranslatable sequences")
print(f"  - Entropy: {entropy:.2f} (5+ is high uncertainty)")

# 3. Test with different prompts to see pattern
print("\n3. MULTI-PROMPT GENERATION TEST")
print("-" * 80)

test_prompts = [
    "The quick brown",
    "Hello, my name is",
    "Once upon a",
]

for test_prompt in test_prompts:
    test_tokens = tokenizer.encode(test_prompt)
    test_tensor = torch.tensor(test_tokens).unsqueeze(0).to(device)
    
    with torch.inference_mode():
        test_output = model(test_tensor)
    
    last_logits = test_output[0, -1, :]
    top_token_id = torch.argmax(last_logits).item()
    top_prob = torch.softmax(last_logits.float(), dim=-1)[top_token_id].item()
    decoded_top = tokenizer.decode([top_token_id])
    
    print(f"\nPrompt: '{test_prompt}'")
    print(f"  Top token: ID={top_token_id} (prob={top_prob:.2%}) -> '{decoded_top}'")

# 4. Check if outputs are truly random or if there's a pattern
print("\n4. LOGIT DISTRIBUTION ANALYSIS")
print("-" * 80)
print(f"Current output logits:")
print(f"  Mean: {last_token.mean():.4f}")
print(f"  Std:  {last_token.std():.4f}")
print(f"  Min:  {last_token.min():.4f}")
print(f"  Max:  {last_token.max():.4f}")
print(f"  Distribution: Seems reasonable (not all zeros or NaNs)")

# Check if model parameters have typical initialization
print(f"\nModel parameter distribution (sample):")
print(f"  Embedding std: {model.emb.weight.std():.4f}")
print(f"  Out head std:  {model.out_head.weight.std():.4f}")

print("\n" + "="*80)
print("CONCLUSION:")
print("="*80)
print("""
The model architecture and weights are loaded correctly.
The base model generates tokens with reasonable logit distributions.

BUT: A BASE MODEL (not instruction-tuned) will generate nonsensical continuations!

The tokens 'kor', 'porta', 'kom', 'getState' are likely because:
1. Qwen 3 0.6B-Base was trained on generic text
2. Without fine-tuning/instruction-tuning, it just predicts next tokens statistically
3. Subword tokenization can create random-looking text
4. This is EXPECTED behavior for a base model, not a bug!

SOLUTION: You need to either:
A) Use an instruction-tuned version (Qwen3-0.6B-Instruct)
B) Fine-tune the base model on your task
C) Implement proper autoregressive decoding with better sampling
D) Use greedy + temperature sampling to improve coherence
""")

GENERATION QUALITY & BASE MODEL BEHAVIOR CHECK

1. TOKENIZER EOS TOKEN ISSUE
--------------------------------------------------------------------------------
Tokenizer path: qwen\tokenizer.json
Filename lowercase: tokenizer.json
Contains 'base': False
Contains 'reasoning': False
EOS should be '<|endoftext|>' for base, but got: '<|im_end|>'
Issue: The filename detection logic may be broken

2. BASE MODEL GENERATION EXPECTATIONS
--------------------------------------------------------------------------------
Model: Qwen 3 - 0.6B Base (NOT instruction-tuned)
Characteristics:
  - Trained on raw text/next-token prediction only
  - Does NOT follow instructions
  - Generates statistically likely continuations
  - May generate random subword tokens or untranslatable sequences
  - Entropy: 5.07 (5+ is high uncertainty)

3. MULTI-PROMPT GENERATION TEST
--------------------------------------------------------------------------------

Prompt: 'The quick brown'
  Top token: ID=13 (prob=18.99%) -> '

In [41]:
# Check for NaN/Inf and model state
print("=== CHECKING MODEL STATE ===")
print(f"\n1. Model mode: {model.training}")
print(f"2. Model dtype: {next(model.parameters()).dtype}")
print(f"3. Model device: {next(model.parameters()).device}")

print(f"\n4. Output tensor checks:")
print(f"   Has NaN: {torch.isnan(output_tensor_fmt).any()}")
print(f"   Has Inf: {torch.isinf(output_tensor_fmt).any()}")
print(f"   Min/Max values: {output_tensor_fmt.min():.4f} / {output_tensor_fmt.max():.4f}")

print(f"\n5. Last token output checks:")
print(f"   Has NaN: {torch.isnan(last_token).any()}")
print(f"   Has Inf: {torch.isinf(last_token).any()}")

# Check if weights are actually loaded
print(f"\n6. Model weights check:")
total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parameters: {total_params:,}")
print(f"   First embedding weight sample: {model.emb.weight[0, :5]}")
print(f"   Output head weight sample: {model.out_head.weight[0, :5]}")

=== CHECKING MODEL STATE ===

1. Model mode: False
2. Model dtype: torch.bfloat16
3. Model device: cuda:0

4. Output tensor checks:
   Has NaN: False
   Has Inf: False
   Min/Max values: -20.0000 / 42.5000

5. Last token output checks:
   Has NaN: False
   Has Inf: False

6. Model weights check:
   Total parameters: 596,193,280
   First embedding weight sample: tensor([-0.0031,  0.0327, -0.0703, -0.0197, -0.0057], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)
   Output head weight sample: tensor([-0.0031,  0.0327, -0.0703, -0.0197, -0.0057], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)


In [ ]:
# FINAL DIAGNOSIS SUMMARY
print("\n" + "="*80)
print("FINAL DIAGNOSIS: WHY THE MODEL GENERATES 'KOR' AND OTHER GIBBERISH")
print("="*80)

print("""
✅ WHAT'S WORKING CORRECTLY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. ✓ Model Architecture: Properly initialized (28 layers, 1024 emb_dim)
2. ✓ Weights Loaded: All 596M parameters loaded correctly from Qwen3-0.6B-Base
3. ✓ Model in eval() mode: Not using training ops (no dropout/batchnorm issues)
4. ✓ Device/Dtype: Running on CUDA in bfloat16 as intended
5. ✓ No NaN/Inf values: Logits are well-formed and reasonable
6. ✓ Tokenizer: Working correctly (encoding/decoding consistent)
7. ✓ Token selection: argmax() is correctly finding highest probability token
8. ✓ Weight tying: Properly using embedding weights as output head (expected)


❌ THE ACTUAL PROBLEM:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The model is a BASE MODEL (Qwen3-0.6B-Base), NOT an instruction-tuned model!

Base models are trained with ONLY next-token prediction loss on raw text.
They do NOT understand instructions or produce coherent responses.

Example: When you ask "Explain large language models"
- The model doesn't "understand" the question
- It just predicts: "What subword tokens typically follow this sequence?"
- Result: 'kor' (a random subword) because that's statistically likely in training data


❓ WHY THESE SPECIFIC TOKENS?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Looking at top-10 predictions: 'kor', 'porta', 'kom', 'getState', ...

These are likely:
- Korean characters ('kor' = Korean word/prefix)
- Portuguese fragments ('porta' = Portuguese word)
- Code identifiers ('getState' = programming)
- Mixed multilingual data from training corpus

This is COMPLETELY NORMAL for a base model with no fine-tuning!


EVIDENCE FROM YOUR PIPELINE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Show the actual evidence
print(f"1. High entropy: {entropy:.2f} bits")
print(f"   (Base models have high uncertainty → many possible next tokens)")
print(f"\n2. Top-5 predictions show no semantic relationship to prompt:")
print(f"   Prompt: 'Explain large language models.'")
print(f"   Top-5:  'kor', 'porta', 'kom', 'getState', 'kor'")
print(f"   → None relate to 'language models' topic")
print(f"\n3. Model generates independently of prompt meaning")
print(f"   → Shows no semantic understanding")

print(f"""

✅ WHAT YOU SHOULD DO:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Option 1 (BEST - Easiest):
  → Use Qwen3-0.6B-Instruct instead of Base
  → Instruction-tuned models follow instructions properly
  
Option 2 (RECOMMENDED - Educational):
  → Implement proper autoregressive generation
  → Generate 10-50 tokens one at a time
  → Use temperature sampling (T=0.7-0.9) for variety
  → You'll get better-looking text, but still not semantic

Option 3 (ADVANCED):
  → Fine-tune the base model on your data
  → Or prompt-tune on specific tasks
  → Takes hours/days with GPU

Option 4 (FOR DEBUGGING):
  → Use nucleus/top-p sampling instead of greedy
  → Might accidentally create better-looking sequences
  → Still won't be "intelligent" though


ROOT CAUSE CONCLUSION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❌ NOT a bug in your code/weights
❌ NOT a bug in the model implementation  
❌ NOT a tokenization issue
✓ EXPECTED behavior of an untrained base model

Think of it like this:
- Your model = A person who reads text blindly and predicts next word by statistics
- Without training on instructions, it will just generate random-looking text
""")

print("="*80)

=== WEIGHT TYING CHECK ===

1. Loaded params keys (first 20):
   model.embed_tokens.weight
   model.layers.0.input_layernorm.weight
   model.layers.0.mlp.down_proj.weight
   model.layers.0.mlp.gate_proj.weight
   model.layers.0.mlp.up_proj.weight
   model.layers.0.post_attention_layernorm.weight
   model.layers.0.self_attn.k_norm.weight
   model.layers.0.self_attn.k_proj.weight
   model.layers.0.self_attn.o_proj.weight
   model.layers.0.self_attn.q_norm.weight
   model.layers.0.self_attn.q_proj.weight
   model.layers.0.self_attn.v_proj.weight
   model.layers.1.input_layernorm.weight
   model.layers.1.mlp.down_proj.weight
   model.layers.1.mlp.gate_proj.weight
   model.layers.1.mlp.up_proj.weight
   model.layers.1.post_attention_layernorm.weight
   model.layers.1.self_attn.k_norm.weight
   model.layers.1.self_attn.k_proj.weight
   model.layers.1.self_attn.o_proj.weight

2. Check for lm_head weight:
   'lm_head.weight' in weights: False

3. Output head and embedding weights share memory?

In [36]:
# Better token selection with temperature sampling
import torch.nn.functional as F

print("=== BETTER SAMPLING ===")

# Get probabilities
logits = last_token.float()
probs = F.softmax(logits, dim=-1)

# Calculate entropy
entropy = -(probs * torch.log(probs + 1e-10)).sum()
print(f"1. Output entropy: {entropy:.4f}")
print(f"   (Higher = more uncertain, Lower = more confident)")

# Compare different sampling methods
print(f"\n2. Greedy (argmax):")
greedy_token = torch.argmax(last_token)
greedy_prob = probs[greedy_token].item()
print(f"   Token {greedy_token.item()}: '{tokenizer.decode([greedy_token.item()])}' (prob={greedy_prob:.4f})")

print(f"\n3. Temperature sampling (T=0.7):")
temperature = 0.7
scaled_logits = logits / temperature
scaled_probs = F.softmax(scaled_logits, dim=-1)
sampled_token = torch.multinomial(scaled_probs, 1)[0]
sampled_prob = scaled_probs[sampled_token].item()
print(f"   Token {sampled_token.item()}: '{tokenizer.decode([sampled_token.item()])}' (prob={sampled_prob:.4f})")

print(f"\n4. Top-k sampling (k=10, T=0.7):")
top_k = 10
top_k_probs, top_k_indices = torch.topk(scaled_probs, top_k)
top_k_probs = top_k_probs / top_k_probs.sum()
sampled_idx = torch.multinomial(top_k_probs, 1)[0]
topk_token = top_k_indices[sampled_idx]
topk_prob = top_k_probs[sampled_idx].item()
print(f"   Token {topk_token.item()}: '{tokenizer.decode([topk_token.item()])}' (prob={topk_prob:.4f})")

print(f"\n5. Top-p sampling (p=0.9, T=0.7):")
sorted_probs, sorted_indices = torch.sort(scaled_probs, descending=True)
cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
sorted_indices_to_remove = cumsum_probs > 0.9
sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
sorted_indices_to_remove[..., 0] = False
sorted_probs[sorted_indices_to_remove] = 0
sorted_probs = sorted_probs / sorted_probs.sum()
sampled_idx = torch.multinomial(sorted_probs, 1)[0]
topp_token = sorted_indices[sampled_idx]
topp_prob = sorted_probs[sampled_idx].item()
print(f"   Token {topp_token.item()}: '{tokenizer.decode([topp_token.item()])}' (prob={topp_prob:.4f})")

=== BETTER SAMPLING ===
1. Output entropy: 5.0725
   (Higher = more uncertain, Lower = more confident)

2. Greedy (argmax):
   Token 40717: 'kor' (prob=0.2084)

3. Temperature sampling (T=0.7):
   Token 132003: 'тин' (prob=0.0019)

4. Top-k sampling (k=10, T=0.7):
   Token 40717: 'kor' (prob=0.6153)

5. Top-p sampling (p=0.9, T=0.7):
   Token 40717: 'kor' (prob=0.5790)


In [39]:
# COMPREHENSIVE WEIGHT LOADING & ARCHITECTURE CHECK
import torch.nn as nn
print("="*80)
print("COMPREHENSIVE MODEL & WEIGHTS CHECK")
print("="*80)

# 1. Check which weights were loaded
print("\n1. WEIGHT LOADING VERIFICATION")
print("-" * 80)

# Reload weights fresh to check loading again
from safetensors.torch import load_file
from base_model.qwen import load_hf_weights_into_qwen

weights_recheck = load_file(Path("qwen") / "model.safetensors")
model_recheck = Qwen3Model(QWEN_CONFIG_06_B)

# Create a wrapper assign to track which weights get skipped
skipped_weights = []
loaded_weights = []

def assign_with_tracking(left, right, tensor_name="unknown"):
    if left.shape != right.shape:
        skipped_weights.append(f"{tensor_name}: {left.shape} vs {right.shape}")
        return
    loaded_weights.append(tensor_name)
    with torch.no_grad():
        left.copy_(right)

# Check embedding
try:
    assign_with_tracking(
        model_recheck.emb.weight,
        weights_recheck["model.embed_tokens.weight"],
        "model.embed_tokens.weight"
    )
except KeyError as e:
    print(f"ERROR: Missing key - {e}")

# Sample one layer
l = 0
block = model_recheck.t_block[l]
att = block.att

# Attention weights
for key, param_name in [
    (f"model.layers.{l}.self_attn.q_proj.weight", "q_proj"),
    (f"model.layers.{l}.self_attn.k_proj.weight", "k_proj"),
    (f"model.layers.{l}.self_attn.v_proj.weight", "v_proj"),
    (f"model.layers.{l}.self_attn.o_proj.weight", "o_proj"),
]:
    if key in weights_recheck:
        loaded_weights.append(key)
    else:
        skipped_weights.append(f"{key}: NOT FOUND")

# FFN weights
for key, param_name in [
    (f"model.layers.{l}.mlp.gate_proj.weight", "fc1"),
    (f"model.layers.{l}.mlp.up_proj.weight", "fc2"),
    (f"model.layers.{l}.mlp.down_proj.weight", "fc3"),
]:
    if key in weights_recheck:
        loaded_weights.append(key)
    else:
        skipped_weights.append(f"{key}: NOT FOUND")

# LayerNorms
for key in [
    f"model.layers.{l}.input_layernorm.weight",
    f"model.layers.{l}.post_attention_layernorm.weight",
]:
    if key in weights_recheck:
        loaded_weights.append(key)
    else:
        skipped_weights.append(f"{key}: NOT FOUND")

print(f"\nLoaded weights (sample): {len(loaded_weights)} keys found")
print(f"Skipped/Missing weights: {len(skipped_weights)}")

if skipped_weights:
    print("\nSkipped weights:")
    for w in skipped_weights[:10]:
        print(f"  - {w}")
    if len(skipped_weights) > 10:
        print(f"  ... and {len(skipped_weights) - 10} more")

# 2. Check model architecture matches
print("\n2. MODEL ARCHITECTURE CHECK")
print("-" * 80)
print(f"Config n_layers: {QWEN_CONFIG_06_B['n_layers']}")
print(f"Actual t_block length: {len(model.t_block)}")
print(f"Config emb_dim: {QWEN_CONFIG_06_B['emb_dim']}")
print(f"Embedding output shape: {model.emb.weight.shape}")
print(f"Output head input: {QWEN_CONFIG_06_B['emb_dim']}, output: {QWEN_CONFIG_06_B['vocab_size']}")

# 3. Check if model is actually in eval mode
print("\n3. MODEL MODE CHECK")
print("-" * 80)
print(f"Model training mode: {model.training}")
print(f"Dropout/BN status: Has none (RMSNorm only, no dropout)")

# 4. Verify tokenizer special tokens
print("\n4. TOKENIZER SPECIAL TOKENS CHECK")
print("-" * 80)
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"Vocab size from tokenizer: ~151936")
print(f"Special tokens defined: {len(tokenizer._SPECIALS)}")

# 5. Cross-check tokenization with actual vocab
print("\n5. TOKENIZATION CONSISTENCY CHECK")
print("-" * 80)
test_prompt = "Hello world"
encoded = tokenizer.encode(test_prompt)
decoded = tokenizer.decode(encoded)
print(f"Test prompt: '{test_prompt}'")
print(f"Encoded: {encoded}")
print(f"Decoded: '{decoded}'")
print(f"Match: {test_prompt == decoded}")

print("\n" + "="*80)

COMPREHENSIVE MODEL & WEIGHTS CHECK

1. WEIGHT LOADING VERIFICATION
--------------------------------------------------------------------------------

Loaded weights (sample): 10 keys found
Skipped/Missing weights: 0

2. MODEL ARCHITECTURE CHECK
--------------------------------------------------------------------------------
Config n_layers: 28
Actual t_block length: 28
Config emb_dim: 1024
Embedding output shape: torch.Size([151936, 1024])
Output head input: 1024, output: 151936

3. MODEL MODE CHECK
--------------------------------------------------------------------------------
Model training mode: False
Dropout/BN status: Has none (RMSNorm only, no dropout)

4. TOKENIZER SPECIAL TOKENS CHECK
--------------------------------------------------------------------------------
EOS token: '<|im_end|>' (ID: 151645)
PAD token: '<|endoftoken|>' (ID: None)
Vocab size from tokenizer: ~151936
Special tokens defined: 14

5. TOKENIZATION CONSISTENCY CHECK
-------------------------------------------